In [3]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multioutput import MultiOutputClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import classification_report

# Load updated dataset
df = pd.read_csv('learning_disability_dataset.csv')

# Encode categorical features safely
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
df['Sleep Quality'] = df['Sleep Quality'].map({'Poor': 0, 'Average': 1, 'Good': 2})

# Prepare features and labels
X = df.drop(columns=['Labels'])

# Clean and split multi-label targets safely
y_raw = df['Labels'].apply(
    lambda x: [
        label.strip().capitalize()
        for label in str(x).split(',')
        if label.strip().lower() not in ['nan', '', 'none']
    ]
)

# Binarize labels
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(y_raw)

# Ensure no "nan"-like class labels in mlb.classes_
valid_classes = [cls for cls in mlb.classes_ if str(cls).lower() not in ['nan', '', 'none']]
mlb.classes_ = np.array(valid_classes)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define models
models = {
    'random_forest': MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42)),
    'svm': MultiOutputClassifier(SVC(probability=True, kernel='linear', random_state=42)),
    'logistic_regression': MultiOutputClassifier(LogisticRegression(max_iter=1000, random_state=42)),
    'knn': MultiOutputClassifier(KNeighborsClassifier(n_neighbors=5)),
    'gradient_boosting': MultiOutputClassifier(GradientBoostingClassifier(n_estimators=100, random_state=42)),
}

# Create output directory
os.makedirs('saved_models', exist_ok=True)

# Train and save models
for name, model in models.items():
    print(f"\n🔹 Training {name} model...")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_pred = np.nan_to_num(y_pred, nan=0.0).astype(int)

    print(f"📊 Classification Report for {name}:")
    try:
        print(classification_report(
            y_test,
            y_pred,
            target_names=mlb.classes_,
            labels=range(len(mlb.classes_))
        ))
    except ValueError as e:
        print("⚠️ Could not generate classification report:", str(e))

    # Save model
    joblib.dump(model, f'saved_models/{name}_model.pkl')

# Save label binarizer
joblib.dump(mlb, 'saved_models/mlb.pkl')

print("\n✅ All models trained and saved in 'saved_models/' folder.")
# Save feature importances if available




🔹 Training random_forest model...
📊 Classification Report for random_forest:
              precision    recall  f1-score   support

        Adhd       0.89      0.83      0.86        70
    Dyslexia       0.99      0.84      0.91        97

   micro avg       0.95      0.83      0.89       167
   macro avg       0.94      0.83      0.88       167
weighted avg       0.95      0.83      0.89       167
 samples avg       0.57      0.53      0.54       167



C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:15


🔹 Training svm model...
📊 Classification Report for svm:
              precision    recall  f1-score   support

        Adhd       0.83      0.83      0.83        70
    Dyslexia       0.90      0.84      0.87        97

   micro avg       0.87      0.83      0.85       167
   macro avg       0.86      0.83      0.85       167
weighted avg       0.87      0.83      0.85       167
 samples avg       0.55      0.53      0.53       167


🔹 Training logistic_regression model...


C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:15

📊 Classification Report for logistic_regression:
              precision    recall  f1-score   support

        Adhd       0.84      0.83      0.83        70
    Dyslexia       0.92      0.84      0.88        97

   micro avg       0.89      0.83      0.86       167
   macro avg       0.88      0.83      0.86       167
weighted avg       0.89      0.83      0.86       167
 samples avg       0.56      0.53      0.53       167


🔹 Training knn model...
📊 Classification Report for knn:
              precision    recall  f1-score   support

        Adhd       0.79      0.74      0.76        70
    Dyslexia       0.87      0.81      0.84        97

   micro avg       0.83      0.78      0.81       167
   macro avg       0.83      0.78      0.80       167
weighted avg       0.83      0.78      0.81       167
 samples avg       0.52      0.50      0.50       167


🔹 Training gradient_boosting model...


C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:15

📊 Classification Report for gradient_boosting:
              precision    recall  f1-score   support

        Adhd       0.95      0.80      0.87        70
    Dyslexia       0.98      0.82      0.89        97

   micro avg       0.96      0.81      0.88       167
   macro avg       0.96      0.81      0.88       167
weighted avg       0.96      0.81      0.88       167
 samples avg       0.56      0.52      0.53       167


✅ All models trained and saved in 'saved_models/' folder.


C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\towqe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:15